In [ ]:
import geopandas as gpd
import pandas as pd
import requests

from unga80.config import DATA_DIR, ROOT_DIR

pd.options.display.max_columns = None

In [ ]:
url = "https://www.naturalearthdata.com/http//www.naturalearthdata.com/download/110m/cultural/ne_110m_admin_0_countries.zip"
url_populated_places = "https://www.naturalearthdata.com/http//www.naturalearthdata.com/download/110m/cultural/ne_110m_populated_places.zip"


def download_admin_countries(url: str, file_name: str):
    response = requests.get(url, stream=True)
    with open(DATA_DIR / file_name, "wb") as f:
        f.write(response.content)

    if (DATA_DIR / file_name).exists():
        print(f"{file_name} downloaded...")
    else:
        print("No file")


# download_admin_countries(url, "ne_110m_admin_0_countries.zip")
download_admin_countries(url_populated_places, "ne_110m_populated_places.zip")

In [ ]:
geo_data = gpd.read_file(DATA_DIR / "ne_110m_admin_0_countries.zip")
gpd_population = gpd.read_file(DATA_DIR / "ne_110m_populated_places.zip")

In [ ]:
geo_data.head()
for i, r in geo_data.iterrows():
    if "and" in str(r["ADMIN"]).lower():
        print(r["ADMIN"])

In [ ]:
geo_data[geo_data["ADM0_A3"].isin(country_info[0]["borders"])].ADMIN.to_list())}

In [ ]:
country_option = "Chile"
print(f'{geo_data[geo_data["ADMIN"]==country_option]["CONTINENT"].values[0]}')
print(
    f'(Economy) {geo_data[geo_data["ADMIN"]==country_option]["ECONOMY"].values[0].split(". ")[-1]}'
)
print(
    f'(Income group) {geo_data[geo_data["ADMIN"]==country_option]["INCOME_GRP"].values[0].split(". ")[-1]}'
)
print(
    f'Population: {geo_data[geo_data["ADMIN"]==country_option]["POP_EST"].apply(int).values[0]:,} (Est. {geo_data[geo_data["ADMIN"]==country_option]["POP_YEAR"].values[0]})'
)
print(
    f'GDP: USD${geo_data[geo_data["ADMIN"]==country_option]["GDP_MD"].apply(int).values[0]:,}M ({geo_data[geo_data["ADMIN"]==country_option]["GDP_YEAR"].apply(int).values[0]})'
)

In [ ]:
geo_data[geo_data["ADMIN"] == country_option].plot()

In [ ]:
gpd_population[gpd_population["ADM0NAME"] == "Chile"]

In [ ]:
gpd_population.head()

In [ ]:
gpd_population[
    [
        "NAME",
        "ADM0NAME",
        "ADM0_A3",
        "LATITUDE",
        "LONGITUDE",
        "POP_MAX",
        "POP_MIN",
        "POP_OTHER",
        "TIMEZONE",
        "geometry",
    ]
].head()

In [ ]:
"AND" in geo_data["ADM0_A3"].unique()

In [ ]:
import altair as alt
from altair.datasets import data

# Data generators for the background
sphere = alt.sphere()
graticule = alt.graticule()

# Source of land data
source = alt.topo_feature(data.world_110m.url, "countries")

# Layering and configuring the components
alt.layer(
    alt.Chart(sphere).mark_geoshape(fill="lightblue"),
    alt.Chart(graticule).mark_geoshape(stroke="white", strokeWidth=0.5),
    alt.Chart(source).mark_geoshape(fill="ForestGreen", stroke="black"),
).project("naturalEarth1").properties(width=600, height=400).configure_view(stroke=None)

In [ ]:
alt.topo_feature(data.world_110m.url, "countries")

In [ ]:
gdf_sel = geo_data[geo_data["ADMIN"] == "Chile"]
alt.Chart(gdf_sel).mark_geoshape()

In [ ]:
fig = px.choropleth(
    locations=["ATG"],
    locationmode="ISO-3",
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.choropleth(
    locations=["USA", "CAN", "MEX", "BRA", "RUS"],
    locationmode="ISO-3",
    color=[100, 85, 72, 95, 68],
    color_continuous_scale="Viridis",
    title="Choropleth with ISO-3 Country Codes",
)
fig.show()

In [ ]:
fig = px.choropleth(
    locations=["ATG"],
    locationmode="ISO-3",
    color=[100],
    color_continuous_scale="Viridis",
    # title="Choropleth with ISO-3 Country Codes",
)
fig.show()

In [ ]:
fig = px.choropleth(
    locations=["Chile"],
    locationmode="country names",
    center={
        "lat": float(
            geo_data[geo_data["ADMIN"] == "Chile"][["LABEL_X", "LABEL_Y"]].values[0][0]
        ),
        "lon": float(
            geo_data[geo_data["ADMIN"] == "Chile"][["LABEL_X", "LABEL_Y"]].values[0][1]
        ),
    },
)
fig.show()

In [ ]:
float(geo_data[geo_data["ADMIN"] == "Chile"][["LABEL_X", "LABEL_Y"]].values[0][0])

In [ ]:
def get_country_information(iso_3: str) -> dict:
    """ADM0_A3 is iso_3"""
    country_rest = f"https://restcountries.com/v3.1/alpha/{iso_3}"
    response = requests.get(country_rest)
    if response.ok:
        return response.json()
    else:
        return {}


response = get_country_information("AND")

In [ ]:
print(response[0]["flag"])
print(response[0]["area"])
print(response[0].get("area"))
print(response[0]["capital"][0])
# print(response[0]["timezones"])
print(", ".join([x for x in response[0]["timezones"]]))
print(response[0]["currencies"])
print(", ".join([x for x in response[0]["currencies"].keys()]))
print(response[0]["languages"])
print(", ".join([response[0]["languages"][x] for x in response[0]["languages"].keys()]))
print(response[0].get("gini"))
if response[0].get("gini"):
    print(
        [f'Gini: {response[0]["gini"][x]} ({x})' for x in response[0]["gini"].keys()][0]
    )
print(response[0]["flags"])
print(response[0]["flags"]["png"])
print(response[0]["flags"]["alt"])
print(response[0]["coatOfArms"])
print(response[0]["coatOfArms"]["png"])
print(response[0]["tld"])
print(response[0]["unMember"])
# print(response[0]["borders"])
print(response[0].get("borders"))

In [ ]:
response

In [ ]:
", ".join(geo_data[geo_data["ADM0_A3"].isin(response[0]["borders"])].ADMIN.to_list())